# load_openaire_researchproduct_authors

Prototipo del nodo `load_openaire_researchproduct_authors` del pipeline `load_openaire`. No guarda datasets.


In [ ]:
from datetime import date
import pandas as pd

%load_ext kedro.ipython


In [ ]:
df_researchproduct_raw = catalog.load('raw/openaire/researchproduct/parquet/researchproduct_dev')
df_researchproduct_raw.head(2)


In [ ]:
def _add_openaire_extracted_metadata(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in _EXTRACTED_META_COLS:
        if col not in df.columns:
            df[col] = pd.NA
    return df


In [ ]:
def _add_openaire_loaded_metadata(df: pd.DataFrame, load_datetime=None) -> pd.DataFrame:
    df = df.copy()
    if load_datetime is None:
        load_datetime = date.today()
    df["_load_datetime"] = load_datetime
    return df


In [ ]:
def load_openaire_researchproduct_authors(df: pd.DataFrame)-> pd.DataFrame:
    df = _add_openaire_extracted_metadata(df)

    df_research_author = df[['id', 'authors', *_EXTRACTED_META_COLS]].explode('authors').reset_index(drop=True)

    df_authors = pd.json_normalize(df_research_author['authors'])

    df_research_author = pd.concat(
        [df_research_author[['id', *_EXTRACTED_META_COLS]].reset_index(drop=True), df_authors.reset_index(drop=True)],
        axis=1,
    )

    df_research_author = _add_openaire_loaded_metadata(df_research_author)

    return df_research_author


In [ ]:
df_researchproduct_author = load_openaire_researchproduct_authors(df_researchproduct_raw)


In [ ]:
pd.DataFrame([{'dataset': 'df_researchproduct_author', 'rows': len(df_researchproduct_author), 'columns': len(df_researchproduct_author.columns)}])


In [ ]:
df_researchproduct_author.head(2)
